# Day 1 — Summation notation $\sum$ (and the moves papers make with it)

Welcome! 🌱 This is day 1 of a slow, friendly, 30-minutes-a-day climb toward reading ML papers. We assume you're comfortable with basic algebra (variables, fractions, powers) — but nothing beyond.

**How each day works:**
1. **Review** — quick recalls from earlier days (today: nothing yet).
2. **Lecture note** — read this. It teaches one small, *usable* idea.
3. **Exercises** — fill in each `# YOUR CODE HERE` stub, then run the test cell below it until it prints ✅.
4. **Solutions** at the very bottom if you get stuck.

No rush. The goal is to *enjoy* it and come back tomorrow.

## 1. Review (~5 min)

It's day 1 — there is nothing to review yet. From day 2 on, this section quietly re-tests older ideas on a spaced schedule so they actually stick. Straight to the lecture!

In [ ]:
import numpy as np

rng = np.random.default_rng(1)  # deterministic randomness for the tests

## 2. Lecture note (~10 min): $\sum$ and three moves you'll use constantly

The big Greek **sigma**, $\sum$, just means **"add a bunch of things up."** Once it stops looking scary, a huge fraction of a paper's equations become readable.

**Anatomy** (read it out loud, left to right):

$$\sum_{i=1}^{n} x_i \;=\; x_1 + x_2 + \dots + x_n$$

- $\sum$ — "the sum of…"
- $i = 1$ under it — start a counter $i$ at 1 ($i$ is just a name; also $j, k, t$).
- $n$ on top — stop when $i$ reaches $n$.
- $x_i$ — the thing added; the subscript means "the $i$-th item of list $x$."

The right-hand piece is a *recipe* in $i$: e.g. $\sum_{i=1}^{3} i^2 = 1 + 4 + 9 = 14$. And the **mean** is just a sum with $\tfrac1n$ in front: $\bar x = \tfrac1n\sum_i x_i$.

That's the symbol. Now the three *moves* — these are what actually lets you follow derivations:

**Move 1 — Linearity (pull constants out, split sums apart).** A sum is just addition, so:

$$\sum_{i=1}^{n}\big(a\,x_i + b\,y_i\big) \;=\; a\sum_{i=1}^{n} x_i \;+\; b\sum_{i=1}^{n} y_i.$$

Constants $a, b$ slide out front; a sum of a sum splits into two sums. Papers do this constantly to simplify a loss.

**Move 2 — Summing over a grid (rows, columns, everything).** For a table $A$ with entry $A_{ij}$ (row $i$, column $j$, shape $m\times n$):

- $\displaystyle\sum_{j=1}^{n} A_{ij}$ = the sum of **row $i$** (we fixed $i$, ran $j$ across the columns).
- $\displaystyle\sum_{i=1}^{m} A_{ij}$ = the sum of **column $j$**.
- $\displaystyle\sum_{i=1}^{m}\sum_{j=1}^{n} A_{ij}$ = **every** entry.

In NumPy these are `A.sum(axis=1)`, `A.sum(axis=0)`, and `A.sum()` — the `axis` is just "which index do I sum over." Getting fluent with *which axis* is half of reading tensor math.

**Move 3 — A double sum can sometimes factor.** This identity shows up in normalizers and expectations:

$$\sum_{i=1}^{m}\sum_{j=1}^{n} x_i\, y_j \;=\; \Big(\sum_{i=1}^{m} x_i\Big)\Big(\sum_{j=1}^{n} y_j\Big).$$

*Why:* inside the inner sum over $j$, the factor $x_i$ is a constant, so it pulls out (Move 1): $\sum_j x_i y_j = x_i\sum_j y_j$. Then $\sum_j y_j$ is itself a constant w.r.t. $i$, so it pulls out of the outer sum. A nested $m\times n$ computation collapses to two cheap sums — that's the kind of simplification a paper will do in one silent step.

## 3. Exercises (~15 min)

### Exercise 1 — linearity

Implement $\displaystyle f(a,b,x,y) = \sum_{i=1}^{n}\big(a\,x_i + b\,y_i\big)$ for scalars `a, b` and equal-length 1-D arrays `x, y`. Return a float. (The test confirms it equals $a\sum x_i + b\sum y_i$ — Move 1.)

In [ ]:
def combo_sum(a, b, x, y):
    # Sum over i of (a * x_i + b * y_i).
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e1(fn):
    for _ in range(5):
        n = int(rng.integers(1, 20))
        a, b = rng.normal(), rng.normal()
        x, y = rng.normal(size=n), rng.normal(size=n)
        got = fn(a, b, x, y)
        # loop oracle
        expected = sum(a * x[i] + b * y[i] for i in range(n))
        assert np.allclose(got, expected), f"got {got}, expected {expected}"
        # and the linearity identity itself
        assert np.allclose(got, a * np.sum(x) + b * np.sum(y))
    print("✅ Exercise 1 passed")

check_e1(combo_sum)

### Exercise 2 — sum over the right axis

For a 2-D array `A` of shape `(m, n)`, return a tuple `(row_sums, col_sums)` where `row_sums[i]` $= \sum_j A_{ij}$ (shape `(m,)`) and `col_sums[j]` $= \sum_i A_{ij}$ (shape `(n,)`). This is all about picking the correct `axis`.

In [ ]:
def margins(A):
    # Return (row_sums, col_sums): sum across columns, then sum across rows.
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e2(fn):
    for _ in range(5):
        m, n = int(rng.integers(1, 6)), int(rng.integers(1, 6))
        A = rng.normal(size=(m, n))
        rs, cs = fn(A)
        rs, cs = np.asarray(rs), np.asarray(cs)
        assert rs.shape == (m,), f"row_sums shape {rs.shape}, expected {(m,)}"
        assert cs.shape == (n,), f"col_sums shape {cs.shape}, expected {(n,)}"
        assert np.allclose(rs, A.sum(axis=1))
        assert np.allclose(cs, A.sum(axis=0))
        # invariant: rows and columns sum to the same grand total
        assert np.allclose(rs.sum(), cs.sum())
    print("✅ Exercise 2 passed")

check_e2(margins)

### Exercise 3 — factor a double sum (the "aha")

Compute $\displaystyle \sum_{i=1}^{m}\sum_{j=1}^{n} x_i\, y_j$ — but using **Move 3**, *without* forming the full $m\times n$ grid. So your function should be just two 1-D sums multiplied. The test compares you against the honest grid computation `np.outer(x, y).sum()`.

In [ ]:
def double_sum_factored(x, y):
    # Return sum_i sum_j x_i y_j WITHOUT building the m x n outer product.
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e3(fn):
    for _ in range(5):
        m, n = int(rng.integers(1, 6)), int(rng.integers(1, 6))
        x, y = rng.normal(size=m), rng.normal(size=n)
        got = fn(x, y)
        grid_oracle = np.outer(x, y).sum()  # the honest double sum
        assert np.allclose(got, grid_oracle), f"got {got}, oracle {grid_oracle}"
    # sanity: ones -> m*n
    assert np.allclose(fn(np.ones(3), np.ones(4)), 12.0)
    print("✅ Exercise 3 passed")

check_e3(double_sum_factored)

## Solutions (collapsed — peek only if stuck)

`ref_`-prefixed reference answers. Running the next cell validates all three against the checks above.

In [ ]:
def ref_combo_sum(a, b, x, y):
    # Move 1 makes this a one-liner, but a loop is equally fine.
    return a * np.sum(x) + b * np.sum(y)

def ref_margins(A):
    # axis=1 collapses columns -> one number per row; axis=0 collapses rows.
    return A.sum(axis=1), A.sum(axis=0)

def ref_double_sum_factored(x, y):
    # Move 3: (sum of x) * (sum of y), no m x n grid.
    return np.sum(x) * np.sum(y)

check_e1(ref_combo_sum)
check_e2(ref_margins)
check_e3(ref_double_sum_factored)
print("\nAll reference solutions pass. 🎉  See you on day 2!")